# Suite CTX — Contexts: the context a procedure takes, as schema data

A context is the standardized argument of a procedure, validation among them, held as schema data
([CAP-319: contexts as schema data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-319)):
a `Contexts.OfContext` value holds `variables`, `symbols` and `parameters`, each keyed by name. A `Contexts.OfVariable`
gives a variable its schema and its value; a `Contexts.OfSymbol` gives a symbol an object schema and the object it
binds; a `Contexts.OfParameter` gives a parameter its schema and either the object it is bound to or the value
assigned to it. Each refers to what it holds by reference: a schema by a `References.OfSchema`, an object by a
`References.OfObject`, a value by a `References.OfValue`
([CAP-320: schema and value references](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-320)).
A sub-context is copied: `Contexts.within(store, outer, own)` gives a new context that holds the outer context's
entries, each entry of its own replacing the outer entry of its name, and nothing links it to either.
`Contexts.validate(store, context)` reports a context's problems, a symbol whose schema is not an object schema among
them. Validation reads a context's parameters (suite PAR, PAR-09..12).

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import json

from mbse.Schemas.Framework import JSON, Builders as B, Contexts, Paths, Plain, References, Validators
from support import raises


def parts(txn):
    """pcb.Chip, a singleton; pcb.Board, which feeds chips; pcb.Part, a union of the two."""
    S = txn.Schemas
    return txn.uid("ctx.parts/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Chip": S.OfObject, "Board": S.OfObject, "Feeds": S.OfRelation, "Part": S.OfUnion})
        .spec([
            lambda _: _.Feeds.name("pcb.Feeds").links([lambda t: t.name("board"), lambda t: t.name("chip")]),
            lambda _: _.Chip.name("pcb.Chip").ref(True).singleton("pcb.TheChip").adjacencies([
                lambda a: a.relation([_.Feeds]).me("chip")]),
            lambda _: _.Board.name("pcb.Board").ref(True).properties({"size": S.OfInt}).adjacencies({
                "feeds": lambda a: a.relation([_.Feeds]).me("board")}),
            lambda _: _.Part.name("pcb.Part").branches([lambda t: t.name("chip").schema([_.Chip]),
                                                       lambda t: t.name("board").schema([_.Board])]),
        ])]))])


store = B.Store()
store.transaction(parts)
Chip, Board, Part, Int = (store.registered(name) for name in ("pcb.Chip", "pcb.Board", "pcb.Part", "Schemas.OfInt"))
OfContext = store.registered("Contexts.OfContext")


def at(schema):
    """A schema object's reference (`References.OfSchema`) to `schema`, through the builders."""
    return lambda r: r.path(Paths.of(schema))


def plain(value):
    return Plain.ToPlain(store)(value.as_schema(), value)


def schema_path(*names):
    """A schema object's path, from the root module through each module and the schema, by name."""
    nested = {"target": {}}
    for name in reversed(names):
        nested = {"entry": {"i": "schemas", "o": "", "properties": {"name": name}, "next": nested}}
    return {"path": {"singleton": ".", "next": nested}}

<a id="CTX-01"></a>
## CTX-01 · Every store holds the context schemas, its own, value objects: a context's variables, symbols and parameters, each keyed by name and each a name, a schema object's reference and what it holds by reference; Contexts.install, run again, writes nothing

In [ ]:
OfVariable, OfSymbol, OfParameter = (store.registered(f"Contexts.{name}") for name in ("OfVariable", "OfSymbol", "OfParameter"))
OfSchema, OfObject, OfValue = (store.registered(f"References.{name}") for name in ("OfSchema", "OfObject", "OfValue"))
assert list(OfVariable.properties) == ["name", "schema", "value"] and list(OfSymbol.properties) == ["name", "schema", "object"]
assert list(OfParameter.properties) == ["name", "schema", "object", "value"]
assert [OfVariable.properties[p].schema for p in ("schema", "value")] == [OfSchema, OfValue]
assert [OfParameter.properties[p].schema for p in ("schema", "object", "value")] == [OfSchema, OfObject, OfValue]
assert OfSymbol.properties["object"].schema is OfObject
assert list(OfContext.properties) == ["variables", "symbols", "parameters"]
for name, element in (("variables", OfVariable), ("symbols", OfSymbol), ("parameters", OfParameter)):
    container = OfContext.properties[name].schema  # keyed by name
    assert container.index is store.Schemas.OfStr.as_schema() and container.element is element
assert not any(schema.ref for schema in (OfVariable, OfSymbol, OfParameter, OfContext))  # value objects
assert B.Store().registered("Contexts.OfContext") is not OfContext  # each store's its own
names = {name: store.registered(name) for name in store.names()}
store.transaction(Contexts.install)  # each found by its name
assert {name: store.registered(name) for name in store.names()} == names

<a id="CTX-02"></a>
## CTX-02 · A context is built through the builders' fluent calls, written as JSON and read back the same; its entries are read by name, each as its visitor

In [ ]:
with store.transaction():
    chip = References.of(store, store.singleton("pcb.TheChip"))
    context = store.Contexts.OfContext(lambda c: c
        .variables({"x": lambda v: v.name("x").schema(at(Int)).value(lambda r: r.value(3))})
        .symbols({"c": lambda s: s.name("c").schema(at(Chip)).object(chip)})
        .parameters({"n": lambda p: p.name("n").schema(at(Int)).value(lambda r: r.value(4)),
                     "h": lambda p: p.name("h").schema(at(Chip)).object(chip)}))
    written = {
        "variables": {"x": {"name": "x", "schema": schema_path("Schemas", "OfInt"), "value": {"value": {"int": 3}}}},
        "symbols": {"c": {"name": "c", "schema": schema_path("pcb", "Chip"), "object": plain(chip)}},
        "parameters": {"n": {"name": "n", "schema": schema_path("Schemas", "OfInt"), "value": {"value": {"int": 4}}},
                       "h": {"name": "h", "schema": schema_path("pcb", "Chip"), "object": plain(chip)}}}
    assert plain(context) == written  # a value, written as any value is
    text = JSON.ToJSON(store)(OfContext, context)
    assert plain(JSON.FromJSON(store)(OfContext, text)) == written  # and read back the same
    assert list(Contexts.entries(store, context, "parameters")) == ["n", "h"]  # its entries, by name, as visitors
    assert Contexts.held(Contexts.entries(store, context, "parameters")["h"].value) is None  # a property not set
    assert Contexts.validate(store, context) == []

<a id="CTX-03"></a>
## CTX-03 · A sub-context is copied: `Contexts.within` gives a new context holding the outer context's entries, each entry of its own replacing the outer one of its name, the outer context unchanged; validation reads each context's own parameters; what is not a context is refused

In [ ]:
with store.transaction():
    chip = References.of(store, store.singleton("pcb.TheChip"))
    outer = store.Contexts.OfContext(lambda c: c
        .symbols({"c": lambda s: s.name("c").object(chip)})
        .parameters({"n": lambda p: p.name("n").value(lambda r: r.value(3)), "h": lambda p: p.name("h").object(chip)}))
    own = store.Contexts.OfContext(lambda c: c.parameters({  # what a statement assigns: n anew, and k
        "n": lambda p: p.name("n").value(lambda r: r.value("x")), "k": lambda p: p.name("k").value(lambda r: r.value(5))}))
    before = plain(outer)
    sub = Contexts.within(store, outer, own)
    assert plain(sub) == {"symbols": before["symbols"], "parameters": {
        "n": {"name": "n", "value": {"value": {"str": "x"}}}, "h": before["parameters"]["h"],
        "k": {"name": "k", "value": {"value": {"int": 5}}}}}  # the outer entries, each of its own replacing one
    assert plain(outer) == before and plain(Contexts.within(store, outer, store.Contexts.OfContext(lambda c: c))) == before
    assert plain(Contexts.within(store, store.Contexts.OfContext(lambda c: c), store.Contexts.OfContext(lambda c: c))) == {}
    n = store.References.OfParameter(lambda p: p.name("n"))
    assert Validators.Validate(store).OfScalar(Int, n, outer) == []  # validation reads each context's own parameters
    assert Validators.Validate(store).OfScalar(Int, n, sub) == [
        "the value: parameter 'n' is assigned a value of another schema than its place expects"]
    for wrong in ({"n": 3}, n):
        with raises(TypeError, match="Contexts.within takes a context (Contexts.OfContext), not "):
            Contexts.within(store, outer, wrong)

<a id="CTX-04"></a>
## CTX-04 · `Contexts.validate` reports an entry named otherwise than its key, a symbol whose schema is not an object schema, a schema reference that designates no schema object the store holds or whose path is not a schema object's, and a parameter both bound to an object and assigned a value; what is not a context is refused

In [ ]:
with store.transaction():
    chip = References.of(store, store.singleton("pcb.TheChip"))
    nowhere = lambda r: r.path(lambda p: p.singleton(".").next(lambda x: x.entry(lambda e: e.i("schemas").o("")
        .properties({"name": "nowhere"}).next(lambda y: y.target(lambda t: t)))))
    context = store.Contexts.OfContext(lambda c: c
        .variables({"x": lambda v: v.name("y").schema(at(Part))})  # any schema, a union among them
        .symbols({"u": lambda s: s.name("u").schema(at(Part)).object(chip),  # an object schema's alone
                  "c": lambda s: s.name("c").schema(at(Chip))})
        .parameters({"n": lambda p: p.name("n").schema(nowhere),
                     "b": lambda p: p.name("b").object(chip).value(lambda r: r.value(3)),
                     "s": lambda p: p.name("s").schema(lambda r: r.path(Paths.of(store.singleton("pcb.TheChip"))))}))
    assert Contexts.validate(store, context) == [
        "variable 'x': its entry is named 'y'",
        "symbol 'u': a schema object's reference designates a 'Schemas.OfUnion', where a 'Schemas.OfObject' is expected",
        "parameter 'n': a schema object's reference designates no schema object the store holds",
        "parameter 'b' is both bound to an object and assigned a value",
        "parameter 's': a schema object's reference (References.OfSchema): a schema object's path starts from the root "
        "module, '.', and each step names an entry, {'i': adjacency, 'o': '', 'properties': {'name': name}}"]
    with raises(TypeError, match="Contexts.entries takes a context (Contexts.OfContext), not "):
        Contexts.validate(store, chip)